# Kaggle Multi-Model Multi-Dataset Depth Benchmark

Inference-only benchmark for EagleVision adapted model vs multiple depth baselines.

- Quantitative comparison on depth datasets
- Qualitative comparison on depth datasets and RGB-only datasets
- Per-baseline `improved` boolean tables vs `ours_adapted`

In [ ]:
# Optional if environment misses packages
# !pip -q install -U transformers datasets pandas matplotlib pillow tqdm scipy

import os
import sys
import random
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

from datasets import load_dataset
from transformers import pipeline

In [ ]:
CONFIG = {
    "seed": 7,
    "fast_mode": False,
    "max_samples_per_dataset_fast": 40,
    "max_samples_per_dataset_full": 150,
    "num_qualitative_pairs": 12,
    "output_dir": "outputs/kaggle_multi_depth_benchmark",

    "adapted_checkpoint_path": "/kaggle/input/models/rooonfr/best-adpated-100ep-scannet/pytorch/default/1/best_100ep.pt",
    "dav2_checkpoint_path": "baseline/depth_anything_v2/checkpoints/depth_anything_v2_metric_hypersim_vits.pth",
    "allow_download_if_missing": True,

    "depth_mode": "metric",
    "encoder": "vits",
    "profile": "hypersim",
    "adapter_hidden_channels": 32,
    "normalize_backbone_input": False,

    # task: depth_eval => quantitative+qualitative, rgb_only => qualitative only
    "datasets": [
        {"name":"nyu_depth_v2_hf", "type":"hf_nyu", "split":"validation", "task":"depth_eval", "enabled":True},
        {"name":"kaggle_scannet_2d", "type":"local_scannet_style", "root":"/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d", "task":"depth_eval", "enabled":False},
        {"name":"local_rgbd_pairs", "type":"folder_pairs", "rgb_glob":"data/benchmark_rgbd/rgb/*.png", "depth_glob":"data/benchmark_rgbd/depth/*.png", "depth_scale":1000.0, "task":"depth_eval", "enabled":False},

        # normal vision datasets (qualitative only)
        {"name":"cifar100_val", "type":"hf_rgb", "hf_dataset":"cifar100", "split":"test", "image_key":"img", "task":"rgb_only", "enabled":True},
        {"name":"beans_val", "type":"hf_rgb", "hf_dataset":"beans", "split":"train", "image_key":"image", "task":"rgb_only", "enabled":True},
        {"name":"food101_val", "type":"hf_rgb", "hf_dataset":"food101", "split":"validation", "image_key":"image", "task":"rgb_only", "enabled":True},
    ],

    "models": [
        {"id":"ours_adapted", "kind":"eaglevision_adapted", "enabled":True},
        {"id":"ours_dav2_base", "kind":"eaglevision_base", "enabled":True},

        {"id":"depth_anything_v2_small", "kind":"hf_pipeline", "hf_model":"depth-anything/Depth-Anything-V2-Small-hf", "enabled":True},
        {"id":"depth_anything_v2_base", "kind":"hf_pipeline", "hf_model":"depth-anything/Depth-Anything-V2-Base-hf", "enabled":True},
        {"id":"depth_anything_v2_large", "kind":"hf_pipeline", "hf_model":"depth-anything/Depth-Anything-V2-Large-hf", "enabled":True},
        {"id":"depth_anything_v1_small", "kind":"hf_pipeline", "hf_model":"LiheYoung/depth-anything-small-hf", "enabled":True},
        {"id":"dpt_large", "kind":"hf_pipeline", "hf_model":"Intel/dpt-large", "enabled":True},
        {"id":"zoedepth_nyu_kitti", "kind":"hf_pipeline", "hf_model":"Intel/zoedepth-nyu-kitti", "enabled":True},
        {"id":"midas_dpt_large", "kind":"torchhub_midas", "model_type":"DPT_Large", "enabled":True},
    ],
}

assert CONFIG["dav2_checkpoint_path"]
assert CONFIG["adapted_checkpoint_path"]

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

KAGGLE_WORKING = Path("/kaggle/working")
RUNNING_ON_KAGGLE = KAGGLE_WORKING.exists()
REPO_DIR = (KAGGLE_WORKING / "EagleVision") if (RUNNING_ON_KAGGLE and (KAGGLE_WORKING / "EagleVision").exists()) else Path.cwd().resolve()

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from eaglevision.models.depth.depth_anything_wrapper import DepthAnythingWithAdapter
from eaglevision.models.rt_depthnvs import RoundTripDepthNVS
from eaglevision.engine.checkpointing import load_checkpoint

MAX_SAMPLES = CONFIG["max_samples_per_dataset_fast"] if CONFIG["fast_mode"] else CONFIG["max_samples_per_dataset_full"]
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

OUT_DIR = REPO_DIR / CONFIG["output_dir"]
for rel in [
    "metrics", "qualitative", "qualitative_rgb_only",
    "plots/per_model", "tables/per_model"
]:
    (OUT_DIR / rel).mkdir(parents=True, exist_ok=True)

print("repo:", REPO_DIR)
print("output:", OUT_DIR)
print("max samples per dataset:", MAX_SAMPLES)

In [ ]:
import subprocess

def resolve_ckpt(path_str: str, default_name: str | None = None) -> Path:
    p = Path(path_str)
    if not p.is_absolute():
        p = (REPO_DIR / p).resolve()

    if p.is_file():
        return p

    if p.is_dir():
        if default_name is not None:
            cand = p / default_name
            if cand.is_file():
                return cand
        cands = sorted(list(p.rglob("*.pth")) + list(p.rglob("*.pt")))
        if cands:
            return cands[0]

    raise FileNotFoundError(f"Checkpoint file not found from: {p}")


def ensure_dav2_checkpoint(path_str: str):
    try:
        return resolve_ckpt(path_str, default_name=f"depth_anything_v2_metric_{CONFIG['profile']}_{CONFIG['encoder']}.pth")
    except Exception:
        if not CONFIG["allow_download_if_missing"]:
            raise

    print("DAV2 checkpoint missing; attempting download...")
    cmd = [
        sys.executable, "-m", "baseline.depth_anything_v2", "download",
        "--mode", "all", "--profile", CONFIG["profile"], "--encoder", CONFIG["encoder"],
    ]
    res = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    print(res.stdout[-1200:])
    if res.returncode != 0:
        print(res.stderr[-1200:])
        raise RuntimeError("DAV2 download failed")
    return resolve_ckpt(path_str, default_name=f"depth_anything_v2_metric_{CONFIG['profile']}_{CONFIG['encoder']}.pth")


dav2_ckpt = ensure_dav2_checkpoint(CONFIG["dav2_checkpoint_path"])
adapted_ckpt = resolve_ckpt(CONFIG["adapted_checkpoint_path"], default_name="best_100ep.pt")

print("DAV2:", dav2_ckpt)
print("Adapted:", adapted_ckpt)

In [ ]:
def to_tensor_rgb(image: Image.Image) -> torch.Tensor:
    arr = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1)


def to_depth_2d(pred, target_hw=None) -> np.ndarray:
    if torch.is_tensor(pred):
        pred = pred.detach().cpu().numpy()
    if isinstance(pred, Image.Image):
        pred = np.asarray(pred)

    pred = np.asarray(pred)

    while pred.ndim > 2 and pred.shape[0] == 1:
        pred = pred[0]

    if pred.ndim == 3:
        if pred.shape[-1] in (1, 3, 4):
            pred = pred[..., 0]
        else:
            pred = pred[0]

    if pred.ndim == 1:
        if target_hw is None:
            raise ValueError(f"1D depth output {pred.shape} without target_hw")
        h, w = target_hw
        n = pred.shape[0]
        if n == w:
            pred = np.tile(pred[None, :], (h, 1))
        elif n == h:
            pred = np.tile(pred[:, None], (1, w))
        elif n == h * w:
            pred = pred.reshape(h, w)
        else:
            raise ValueError(f"Cannot reshape 1D depth of length {n} to {(h, w)}")

    if pred.ndim != 2:
        raise ValueError(f"Expected 2D depth map, got shape {pred.shape}")

    return pred.astype(np.float32)


def resize_depth_like_np(pred_2d: np.ndarray, h: int, w: int) -> np.ndarray:
    t = torch.from_numpy(pred_2d).unsqueeze(0).unsqueeze(0).float()
    out = F.interpolate(t, size=(h, w), mode="bilinear", align_corners=False)
    return out[0, 0].cpu().numpy().astype(np.float32)


def valid_mask(depth: np.ndarray) -> np.ndarray:
    return np.isfinite(depth) & (depth > 1e-6)


def median_scale(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray) -> np.ndarray:
    pv = pred[mask]
    gv = gt[mask]
    if pv.size == 0:
        return pred
    s = np.median(gv) / max(np.median(pv), 1e-6)
    return pred * s


def metric_abs_rel(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray) -> float:
    v = np.abs(pred[mask] - gt[mask]) / np.clip(gt[mask], 1e-6, None)
    return float(np.mean(v)) if v.size else np.nan


def metric_rmse(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray) -> float:
    v = (pred[mask] - gt[mask]) ** 2
    return float(np.sqrt(np.mean(v))) if v.size else np.nan


def metric_delta1(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray) -> float:
    p = np.clip(pred[mask], 1e-6, None)
    g = np.clip(gt[mask], 1e-6, None)
    ratio = np.maximum(p / g, g / p)
    return float(np.mean(ratio < 1.25)) if ratio.size else np.nan


def eval_depth_metrics(pred: np.ndarray, gt: np.ndarray) -> dict[str, float]:
    mask = valid_mask(gt)
    if mask.sum() == 0:
        return {
            "abs_rel": np.nan,
            "rmse": np.nan,
            "delta1": np.nan,
            "ms_abs_rel": np.nan,
            "ms_rmse": np.nan,
            "ms_delta1": np.nan,
            "valid_ratio": 0.0,
        }

    raw = {
        "abs_rel": metric_abs_rel(pred, gt, mask),
        "rmse": metric_rmse(pred, gt, mask),
        "delta1": metric_delta1(pred, gt, mask),
    }

    pred_ms = median_scale(pred, gt, mask)
    ms = {
        "ms_abs_rel": metric_abs_rel(pred_ms, gt, mask),
        "ms_rmse": metric_rmse(pred_ms, gt, mask),
        "ms_delta1": metric_delta1(pred_ms, gt, mask),
    }

    return {**raw, **ms, "valid_ratio": float(mask.mean())}


def robust_range(arrs, q_lo=2, q_hi=98):
    vals = []
    for a in arrs:
        m = np.isfinite(a)
        if m.any():
            vals.append(a[m].ravel())
    if not vals:
        return 0.0, 1.0
    x = np.concatenate(vals)
    lo, hi = np.percentile(x, [q_lo, q_hi])
    if (not np.isfinite(lo)) or (not np.isfinite(hi)) or hi <= lo:
        lo, hi = float(np.min(x)), float(np.max(x) + 1e-6)
    return float(lo), float(hi)


def colorize_depth(depth: np.ndarray, vmin=None, vmax=None) -> np.ndarray:
    d = depth.copy()
    m = np.isfinite(d)
    if m.sum() == 0:
        return np.zeros((depth.shape[0], depth.shape[1], 3), dtype=np.uint8)

    if vmin is None or vmax is None:
        lo, hi = np.percentile(d[m], [2, 98])
    else:
        lo, hi = float(vmin), float(vmax)

    d = np.clip((d - lo) / max(hi - lo, 1e-6), 0, 1)
    c = plt.get_cmap("magma")(d)[..., :3]
    return (c * 255).astype(np.uint8)

In [ ]:
class BaseEstimator:
    def predict(self, image: Image.Image) -> np.ndarray:
        raise NotImplementedError


class EagleVisionAdaptedEstimator(BaseEstimator):
    def __init__(self):
        depth_model = DepthAnythingWithAdapter(
            mode=CONFIG["depth_mode"],
            encoder=CONFIG["encoder"],
            profile=CONFIG["profile"],
            checkpoint_path=dav2_ckpt,
            freeze_backbone=True,
            adapter_hidden_channels=CONFIG["adapter_hidden_channels"],
            normalize_backbone_input=CONFIG["normalize_backbone_input"],
        ).to(DEVICE).eval()
        self.model = RoundTripDepthNVS(depth_model).to(DEVICE).eval()
        load_checkpoint(adapted_ckpt, self.model)

    @torch.no_grad()
    def predict(self, image: Image.Image) -> np.ndarray:
        x = to_tensor_rgb(image).unsqueeze(0).to(DEVICE)
        d = self.model.depth_model(x)["adapted_depth"][0, 0]
        return d.detach().cpu().numpy().astype(np.float32)


class EagleVisionBaseEstimator(BaseEstimator):
    def __init__(self):
        self.model = DepthAnythingWithAdapter(
            mode=CONFIG["depth_mode"],
            encoder=CONFIG["encoder"],
            profile=CONFIG["profile"],
            checkpoint_path=dav2_ckpt,
            freeze_backbone=True,
            adapter_hidden_channels=CONFIG["adapter_hidden_channels"],
            normalize_backbone_input=CONFIG["normalize_backbone_input"],
        ).to(DEVICE).eval()

    @torch.no_grad()
    def predict(self, image: Image.Image) -> np.ndarray:
        x = to_tensor_rgb(image).unsqueeze(0).to(DEVICE)
        d = self.model(x)["base_depth"][0, 0]
        return d.detach().cpu().numpy().astype(np.float32)


class HFPipelineEstimator(BaseEstimator):
    def __init__(self, model_id: str):
        self.pipe = pipeline("depth-estimation", model=model_id, device=0 if DEVICE.type == "cuda" else -1)

    def predict(self, image: Image.Image) -> np.ndarray:
        out = self.pipe(image)
        if "predicted_depth" in out:
            d = out["predicted_depth"]
            if torch.is_tensor(d):
                return d.detach().cpu().numpy().astype(np.float32)
            return np.asarray(d, dtype=np.float32)
        d = out["depth"]
        return np.asarray(d, dtype=np.float32)


class MiDaSEstimator(BaseEstimator):
    def __init__(self, model_type: str = "DPT_Large"):
        self.model = torch.hub.load("intel-isl/MiDaS", model_type).to(DEVICE).eval()
        transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
        self.transform = transforms.dpt_transform if "DPT" in model_type else transforms.small_transform

    @torch.no_grad()
    def predict(self, image: Image.Image) -> np.ndarray:
        arr = np.asarray(image.convert("RGB"))
        inp = self.transform(arr).to(DEVICE)
        pred = self.model(inp)
        pred = F.interpolate(pred.unsqueeze(1), size=arr.shape[:2], mode="bicubic", align_corners=False).squeeze()
        return pred.detach().cpu().numpy().astype(np.float32)


def try_build_estimator(spec: dict[str, Any]):
    try:
        k = spec["kind"]
        if k == "eaglevision_adapted":
            return EagleVisionAdaptedEstimator(), None
        if k == "eaglevision_base":
            return EagleVisionBaseEstimator(), None
        if k == "hf_pipeline":
            return HFPipelineEstimator(spec["hf_model"]), None
        if k == "torchhub_midas":
            return MiDaSEstimator(spec.get("model_type", "DPT_Large")), None
        return None, f"Unknown kind {k}"
    except Exception as e:
        return None, repr(e)

In [ ]:
def load_hf_nyu(split: str, max_samples: int):
    ds = None
    errors = []
    candidates = [
        {"path": "sayakpaul/nyu_depth_v2", "kwargs": {"revision": "refs/convert/parquet"}},
        {"path": "sayakpaul/nyu_depth_v2", "kwargs": {}},
    ]

    for c in candidates:
        try:
            ds = load_dataset(c["path"], split=split, **c["kwargs"])
            break
        except Exception as e:
            errors.append(f"{c['path']} {c['kwargs']}: {e}")

    if ds is None:
        raise RuntimeError("Failed NYU load:
" + "
".join(errors))

    n = min(len(ds), max_samples)
    rows = []
    for i in range(n):
        ex = ds[i]
        rows.append({
            "sample_id": f"nyu_{i:06d}",
            "image": ex["image"].convert("RGB"),
            "depth": np.asarray(ex["depth_map"], dtype=np.float32),
        })
    return rows


def load_hf_rgb(dataset_name: str, split: str, image_key: str, max_samples: int):
    ds = load_dataset(dataset_name, split=split)
    n = min(len(ds), max_samples)
    rows = []
    for i in range(n):
        im = ds[i][image_key]
        if not isinstance(im, Image.Image):
            im = Image.fromarray(np.asarray(im))
        rows.append({
            "sample_id": f"{dataset_name.replace('/', '_')}_{i:06d}",
            "image": im.convert("RGB"),
            "depth": None,
        })
    return rows


def load_local_scannet_style(root: str, max_samples: int):
    rows = []
    root = Path(root)
    if not root.exists():
        return rows

    scenes = sorted([p for p in root.glob("*") if p.is_dir()])
    for scene in scenes:
        color_dirs = [scene / "color", scene / "rgb", scene / "images"]
        depth_dirs = [scene / "depth", scene / "depths"]
        cdir = next((d for d in color_dirs if d.exists()), None)
        ddir = next((d for d in depth_dirs if d.exists()), None)
        if cdir is None or ddir is None:
            continue

        colors = sorted(cdir.glob("*.jpg")) + sorted(cdir.glob("*.png"))
        for cp in colors:
            dp_png = ddir / (cp.stem + ".png")
            dp_npy = ddir / (cp.stem + ".npy")
            if dp_png.exists():
                depth = np.asarray(Image.open(dp_png), dtype=np.float32)
                if np.nanmax(depth) > 100:
                    depth = depth / 1000.0
            elif dp_npy.exists():
                depth = np.load(dp_npy).astype(np.float32)
            else:
                continue

            rows.append({
                "sample_id": f"{scene.name}_{cp.stem}",
                "image": Image.open(cp).convert("RGB"),
                "depth": depth,
            })
            if len(rows) >= max_samples:
                return rows

    return rows


def load_folder_pairs(rgb_glob: str, depth_glob: str, depth_scale: float, max_samples: int):
    rgbs = sorted((REPO_DIR).glob(rgb_glob))
    deps = sorted((REPO_DIR).glob(depth_glob))
    n = min(len(rgbs), len(deps), max_samples)
    rows = []
    for i in range(n):
        rows.append({
            "sample_id": rgbs[i].stem,
            "image": Image.open(rgbs[i]).convert("RGB"),
            "depth": np.asarray(Image.open(deps[i]), dtype=np.float32) / float(depth_scale),
        })
    return rows


def load_dataset_rows(spec: dict[str, Any], max_samples: int):
    t = spec["type"]
    if t == "hf_nyu":
        return load_hf_nyu(spec.get("split", "validation"), max_samples)
    if t == "hf_rgb":
        return load_hf_rgb(spec["hf_dataset"], spec.get("split", "train"), spec.get("image_key", "image"), max_samples)
    if t == "local_scannet_style":
        return load_local_scannet_style(spec["root"], max_samples)
    if t == "folder_pairs":
        return load_folder_pairs(spec["rgb_glob"], spec["depth_glob"], spec.get("depth_scale", 1000.0), max_samples)
    raise ValueError(f"Unknown dataset type: {t}")

In [ ]:
enabled_models = [m for m in CONFIG["models"] if m.get("enabled", True)]
enabled_datasets = [d for d in CONFIG["datasets"] if d.get("enabled", True)]

print("Enabled models:")
for m in enabled_models:
    print(" -", m["id"])
print("Enabled datasets:")
for d in enabled_datasets:
    print(" -", d["name"], "task=", d.get("task", "depth_eval"))

estimators = {}
skipped_models = []
for m in enabled_models:
    print("Loading model:", m["id"])
    est, err = try_build_estimator(m)
    if est is None:
        skipped_models.append((m["id"], err))
        print("  [skip]", err)
    else:
        estimators[m["id"]] = est

if not estimators:
    raise RuntimeError("No model loaded successfully")

rows = []
qual_rows = []
qual_rgb_only = []

for dspec in enabled_datasets:
    dname = dspec["name"]
    task = dspec.get("task", "depth_eval")
    print("=== Dataset:", dname, "task:", task, "===")
    try:
        samples = load_dataset_rows(dspec, MAX_SAMPLES)
    except Exception as e:
        print("[skip dataset]", dname, e)
        continue

    print("samples:", len(samples))
    if not samples:
        continue

    qual_idx = set(np.linspace(0, len(samples)-1, min(CONFIG["num_qualitative_pairs"], len(samples))).astype(int).tolist())

    for i, s in enumerate(tqdm(samples, desc=dname)):
        image = s["image"]
        gt = s["depth"]
        if gt is not None:
            gt = gt.astype(np.float32)
            h, w = gt.shape[:2]
        else:
            arr = np.asarray(image)
            h, w = arr.shape[:2]

        preds = {}
        for mid, est in estimators.items():
            raw_pred = est.predict(image)
            pred_2d = to_depth_2d(raw_pred, target_hw=(h, w))
            pred = resize_depth_like_np(pred_2d, h, w)
            preds[mid] = pred

            if gt is not None:
                met = eval_depth_metrics(pred, gt)
                rows.append({"dataset": dname, "sample_id": s["sample_id"], "model": mid, **met})

        if i in qual_idx:
            obj = {
                "dataset": dname,
                "sample_id": s["sample_id"],
                "image": np.asarray(image),
                "gt": gt,
                "preds": preds,
            }
            if gt is None:
                qual_rgb_only.append(obj)
            else:
                qual_rows.append(obj)

per_sample = pd.DataFrame(rows)
if len(per_sample) > 0:
    per_sample_path = OUT_DIR / "metrics" / "per_sample_metrics.csv"
    per_sample.to_csv(per_sample_path, index=False)
    print("saved:", per_sample_path)
    print(per_sample.head())
else:
    print("No quantitative depth rows (only rgb_only datasets may be enabled).")

if skipped_models:
    skipped_path = OUT_DIR / "metrics" / "skipped_models.csv"
    pd.DataFrame(skipped_models, columns=["model", "error"]).to_csv(skipped_path, index=False)
    print("saved:", skipped_path)

In [ ]:
if len(per_sample) == 0:
    print("Skipping quantitative aggregation: no depth GT datasets were evaluated.")
else:
    metric_cols = ["abs_rel", "rmse", "delta1", "ms_abs_rel", "ms_rmse", "ms_delta1", "valid_ratio"]
    summary = per_sample.groupby(["dataset", "model"], as_index=False)[metric_cols].mean(numeric_only=True)
    summary_path = OUT_DIR / "metrics" / "summary_by_dataset_model.csv"
    summary.to_csv(summary_path, index=False)
    print("saved:", summary_path)

    OURS_ID = "ours_adapted"
    higher_better = {"delta1", "ms_delta1", "valid_ratio"}

    comp_rows = []
    for dname in sorted(summary["dataset"].unique()):
        ds = summary[summary["dataset"] == dname].copy()
        ours = ds[ds["model"] == OURS_ID]
        if len(ours) == 0:
            continue
        ours = ours.iloc[0]

        for _, r in ds.iterrows():
            if r["model"] == OURS_ID:
                continue
            for metric in metric_cols:
                ov = float(ours[metric])
                bv = float(r[metric])
                improved = (ov > bv) if metric in higher_better else (ov < bv)
                better_delta = (ov - bv) if metric in higher_better else (bv - ov)
                comp_rows.append({
                    "dataset": dname,
                    "baseline_model": r["model"],
                    "metric": metric,
                    "ours_value": ov,
                    "baseline_value": bv,
                    "delta_ours_minus_baseline": ov - bv,
                    "delta_positive_means_ours_better": better_delta,
                    "improved": bool(improved),
                })

    vs_ours = pd.DataFrame(comp_rows)
    vs_ours_path = OUT_DIR / "metrics" / "vs_ours_metric_comparison.csv"
    vs_ours.to_csv(vs_ours_path, index=False)
    print("saved:", vs_ours_path)

    plot_dir = OUT_DIR / "plots" / "per_model"
    table_dir = OUT_DIR / "tables" / "per_model"
    metrics_focus = ["ms_abs_rel", "ms_rmse", "ms_delta1", "abs_rel", "rmse", "delta1"]

    for (dname, bmodel), g in vs_ours.groupby(["dataset", "baseline_model"]):
        g = g[g["metric"].isin(metrics_focus)].copy().sort_values("metric")
        tpath = table_dir / f"{dname}__{bmodel}__vs_ours.csv"
        g.to_csv(tpath, index=False)

        fig, ax = plt.subplots(figsize=(10, 4.8))
        x = np.arange(len(g))
        vals = g["delta_positive_means_ours_better"].values
        colors = ["#1b9e77" if b else "#d95f02" for b in g["improved"].values]
        ax.bar(x, vals, color=colors)
        ax.axhline(0.0, color="black", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels(g["metric"].tolist(), rotation=30, ha="right")
        ax.set_ylabel("Positive means ours better")
        ax.set_title(f"{dname} | ours vs {bmodel}")
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        ppath = plot_dir / f"{dname}__{bmodel}__vs_ours.png"
        fig.savefig(ppath, dpi=150)
        plt.close(fig)

    print("saved tables:", table_dir)
    print("saved plots:", plot_dir)

## Qualitative Comparison (Depth Datasets)

This cell saves `RGB | GT | Ours(raw) | Baseline(raw) | Ours(ms) | Baseline(ms)` using a shared color scale per row.

In [ ]:
baseline_ids = [mid for mid in estimators.keys() if mid != "ours_adapted"]
qual_root = OUT_DIR / "qualitative"
qual_root.mkdir(parents=True, exist_ok=True)

for bmodel in baseline_ids:
    bdir = qual_root / f"ours_vs_{bmodel}"
    bdir.mkdir(parents=True, exist_ok=True)

    for i, q in enumerate(qual_rows):
        if "ours_adapted" not in q["preds"] or bmodel not in q["preds"]:
            continue

        gt = q["gt"]
        ours = q["preds"]["ours_adapted"]
        base = q["preds"][bmodel]

        m = valid_mask(gt)
        ours_ms = median_scale(ours, gt, m) if m.any() else ours
        base_ms = median_scale(base, gt, m) if m.any() else base

        vmin, vmax = robust_range([gt, ours, base, ours_ms, base_ms])

        fig, axes = plt.subplots(1, 6, figsize=(24, 4))
        axes[0].imshow(q["image"]); axes[0].set_title("RGB"); axes[0].axis("off")
        axes[1].imshow(gt, cmap="magma", vmin=vmin, vmax=vmax); axes[1].set_title("GT"); axes[1].axis("off")
        axes[2].imshow(ours, cmap="magma", vmin=vmin, vmax=vmax); axes[2].set_title("Ours (raw)"); axes[2].axis("off")
        axes[3].imshow(base, cmap="magma", vmin=vmin, vmax=vmax); axes[3].set_title(f"{bmodel} (raw)"); axes[3].axis("off")
        axes[4].imshow(ours_ms, cmap="magma", vmin=vmin, vmax=vmax); axes[4].set_title("Ours (median-scaled)"); axes[4].axis("off")
        axes[5].imshow(base_ms, cmap="magma", vmin=vmin, vmax=vmax); axes[5].set_title(f"{bmodel} (median-scaled)"); axes[5].axis("off")

        fig.suptitle(f"{q['dataset']} | {q['sample_id']} | shared scale [{vmin:.3f}, {vmax:.3f}]")
        fig.tight_layout()
        out = bdir / f"{i:03d}_{q['dataset']}_{q['sample_id']}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        plt.close(fig)

print("saved depth qualitative:", qual_root)

## Qualitative Comparison (RGB-only Datasets)

Saves `RGB + per-model depth maps` for normal vision datasets (no GT depth).

In [ ]:
rgb_root = OUT_DIR / "qualitative_rgb_only"
rgb_root.mkdir(parents=True, exist_ok=True)

model_ids = list(estimators.keys())
for i, q in enumerate(qual_rgb_only):
    cols = 1 + len(model_ids)
    fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))

    axes[0].imshow(q["image"])
    axes[0].set_title("RGB")
    axes[0].axis("off")

    for j, mid in enumerate(model_ids, start=1):
        d = q["preds"][mid]
        vmin, vmax = robust_range([d])
        axes[j].imshow(d, cmap="magma", vmin=vmin, vmax=vmax)
        axes[j].set_title(mid)
        axes[j].axis("off")

    fig.suptitle(f"{q['dataset']} | {q['sample_id']}")
    fig.tight_layout()
    out = rgb_root / f"{i:03d}_{q['dataset']}_{q['sample_id']}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)

print("saved rgb-only qualitative:", rgb_root)

## Final Quantitative Comparison View

In [ ]:
metrics_dir = OUT_DIR / "metrics"
summary_path = metrics_dir / "summary_by_dataset_model.csv"
vs_ours_path = metrics_dir / "vs_ours_metric_comparison.csv"

if not summary_path.exists() or not vs_ours_path.exists():
    print("No quantitative files found yet. Ensure at least one depth_eval dataset is enabled and processed.")
else:
    summary = pd.read_csv(summary_path)
    vs_ours = pd.read_csv(vs_ours_path)

    print("=== Summary by dataset/model ===")
    display(summary.sort_values(["dataset", "ms_abs_rel", "ms_rmse"], ascending=[True, True, True]))

    focus_metrics = ["abs_rel", "rmse", "delta1", "ms_abs_rel", "ms_rmse", "ms_delta1"]
    df = vs_ours[vs_ours["metric"].isin(focus_metrics)].copy()

    rows = []
    for (dataset, baseline), g in df.groupby(["dataset", "baseline_model"]):
        n = len(g)
        wins = int(g["improved"].sum())
        rows.append({
            "dataset": dataset,
            "baseline_model": baseline,
            "metrics_count": n,
            "wins": wins,
            "losses": n - wins,
            "win_rate": wins / n if n else np.nan,
            "mean_delta_positive_means_ours_better": float(g["delta_positive_means_ours_better"].mean()) if n else np.nan,
            "overall_better_than_baseline": bool((wins / n) > 0.5) if n else False,
        })

    verdict = pd.DataFrame(rows).sort_values(["dataset", "win_rate", "mean_delta_positive_means_ours_better"], ascending=[True, False, False])
    print("=== Ours vs each baseline (focus metrics) ===")
    display(verdict)

    verdict_path = metrics_dir / "final_verdict_per_baseline.csv"
    verdict.to_csv(verdict_path, index=False)
    print("saved:", verdict_path)